In [2]:
import os
os.getcwd()

'/Users/ming/Desktop/COM_SCI_245/cs245_project/AgentSocietyChallenge/rec_agent_experiment'

In [3]:
import json
import pandas as pd
from tqdm import tqdm
import os

In [4]:
pd.set_option('display.max_columns', None)
pd.reset_option('display.max_colwidth')

In [5]:
def load_jsonlines(file_path, n_rows=None):
    """
    Load a JSON Lines (.json or .jsonl) file into a Pandas DataFrame safely.

    Args:
        file_path (str): Path to the JSON Lines file
        n_rows (int, optional): Limit number of rows to load (for large files)

    Returns:
        pd.DataFrame: Parsed dataframe
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f, desc=f"Loading {os.path.basename(file_path)}")):
            try:
                record = json.loads(line.strip())
                data.append(record)
            except json.JSONDecodeError:
                continue  # skip malformed lines
            if n_rows and i >= n_rows:
                break
    df = pd.DataFrame(data)
    print(f"✅ Loaded {len(df):,} rows × {len(df.columns)} columns.")
    return df

In [6]:
user_df = load_jsonlines('../data/user.json')
item_df = load_jsonlines('../data/item.json')
review_df = load_jsonlines('../data/review.json')

Loading user.json: 889698it [00:04, 202140.20it/s]


✅ Loaded 889,698 rows × 23 columns.


Loading item.json: 358923it [00:12, 29356.42it/s]


✅ Loaded 358,923 rows × 55 columns.


Loading review.json: 5171890it [00:33, 154148.85it/s]


✅ Loaded 5,171,890 rows × 23 columns.


In [7]:
sampled = (
    item_df
    .groupby(['source', 'type'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 2), random_state=42))
    .reset_index(drop=True)
)
display(sampled[['source', 'type']].head(10))

/var/folders/wg/5j0bq2jj17g_drbmybsk50pr0000gn/T/ipykernel_30894/3201543139.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 2), random_state=42))


,source,type
0,amazon,product
1,amazon,product
2,goodreads,book
3,goodreads,book
4,yelp,business
5,yelp,business


In [8]:
sources = ['yelp','amazon', 'goodreads']
data = {'item': item_df, 'user': user_df, 'review': review_df}

### item df eda

In [9]:
for name, df in data.items():
    for source in sources:
        not_nan_col = df[df.source == source].columns[~df[df.source == source].isna().all()]
        print(f'{name}_{source} columns: {not_nan_col}')

item_yelp columns: Index(['item_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours', 'source', 'type'],
      dtype='object')
item_amazon columns: Index(['item_id', 'categories', 'source', 'type', 'main_category', 'title',
       'average_rating', 'rating_number', 'features', 'description', 'price',
       'images', 'videos', 'store', 'details', 'subtitle', 'author'],
      dtype='object')
item_goodreads columns: Index(['item_id', 'source', 'type', 'title', 'average_rating', 'description',
       'isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'kindle_asin', 'similar_books',
       'format', 'link', 'authors', 'publisher', 'num_pages',
       'publication_day', 'isbn13', 'publication_month', 'edition_information',
       'publication_year', 'url', 'image_url', 'ratings_count', 'work_id',
       'title_w

In [10]:
item_df['type']

0         business
1         business
2         business
3         business
4         business
            ...   
358918        book
358919        book
358920        book
358921        book
358922        book
Name: type, Length: 358923, dtype: object

In [11]:
#in items.json, amazon's categories data is list and yelp's categories data is str
cols_to_check = ['type', 'source', 'categories']

for col in cols_to_check:
    mask = item_df[col].apply(lambda x: isinstance(x, (list, dict, set)))
    if mask.any():  # 如果这一列有问题
        print(f"\n🚨 Column '{col}' has {mask.sum()} unhashable elements:")
        for idx, val in item_df.loc[mask, col].items():
            print(f"  Row {idx}: {val}")


🚨 Column 'categories' has 76047 unhashable elements:
  Row 32869: ['Industrial & Scientific', 'Test, Measure & Inspect', 'Dimensional Measurement', 'Calipers', 'Dial Calipers']
  Row 32870: ['Industrial & Scientific', 'Industrial Electrical', 'Passive Components', 'Resistors', 'Variable Resistors', 'Potentiometers']
  Row 32871: ['Industrial & Scientific', 'Food Service Equipment & Supplies', 'Disposables', 'Take Out Containers', 'Bakery Take Out Containers']
  Row 32872: ['Industrial & Scientific', 'Material Handling Products', 'Industrial Magnets', 'Rare Earth Magnets']
  Row 32873: ['Industrial & Scientific', 'Abrasive & Finishing Products', 'Finishing Products', 'Manual Sanding Products', 'Sanding Sponges']
  Row 32874: ['Industrial & Scientific', 'Professional Medical Supplies', 'Diagnostics & Screening', 'Stethoscopes']
  Row 32875: ['Industrial & Scientific', 'Hydraulics, Pneumatics & Plumbing', 'Fittings', 'Pipe Fittings']
  Row 32876: ['Industrial & Scientific', 'Fasteners', 

In [12]:
yelp_categories = item_df.loc[item_df['source'] == 'yelp',['type','source','categories']].drop_duplicates()
yelp_categories['categories_large'] = yelp_categories['categories'].str.split(', ').str[1]

In [13]:
print('item data: yelp different categories; yelp task-"candidate_category" is type + categories_large')
display(yelp_categories[['type','source','categories_large']].drop_duplicates())
print('item data: amazon,goodreads type and source')
display(item_df.loc[item_df['source'] != 'yelp',['type','source']].drop_duplicates())
print('user data: source')
display(user_df[['source']].drop_duplicates())
print('review data: type and source')
display(review_df[['type','source']].drop_duplicates())

item data: yelp different categories; yelp task-"candidate_category" is type + categories_large


,type,source,categories_large
0,business,yelp,Shopping
1,business,yelp,Food
2,business,yelp,Restaurants
3,business,yelp,Auto Parts & Supplies
5,business,yelp,Bars
...,...,...,...
32041,business,yelp,Shoe Shine
32117,business,yelp,Basketball Courts
32472,business,yelp,Food Banks
32720,business,yelp,Osteopathic Physicians


item data: amazon,goodreads type and source


,type,source
32869,product,amazon
108916,book,goodreads


user data: source


,source
0,yelp
558111,amazon
752327,goodreads


review data: type and source


,type,source
0,business,yelp
1827321,product,amazon
3740357,book,goodreads


In [21]:
yelp_samples = {}

for name, df in data.items():
    # 只保留 Yelp 的数据
    yelp_df = df[df["source"] == "yelp"]

    # 和上面一样，先过滤掉在 Yelp 子集里全是 NaN 的列
    yelp_cols = yelp_df.columns[~yelp_df.isna().all()]

    # 只在这些有效列上取样
    sample = yelp_df[yelp_cols].sample(1, random_state=42)
    yelp_samples[name] = sample

    print(f"=== {name}_yelp sample (filtered columns) ===")
    display(sample)

=== item_yelp sample (filtered columns) ===


,item_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours,source,type
13631,rzLJskZrSPj6hJHOPmJ0TA,Butcher N' Barbeque at Armature Works,1910 N Ola Ave,Tampa,FL,33602,27.961033,-82.463713,2.5,125.0,1.0,"{'Alcohol': 'u'beer_and_wine'', 'HasTV': 'Fals...","Restaurants, Barbeque, Public Markets, Chicken...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-22:0', ...",yelp,business


=== user_yelp sample (filtered columns) ===


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,source
352517,9Fui6guEn9RbroIh_tFlFQ,Rebecca,2.0,2017-09-11 23:15:00,0.0,0.0,1.0,,None,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,yelp


=== review_yelp sample (filtered columns) ===


,review_id,user_id,item_id,stars,useful,funny,cool,text,date,source,type
723221,YjbgzO48fHmkEdukbEWmlw,cLdfK9YF5TztXiI6OBiIQw,xdferXKwbgXAxbtmAShhAQ,3.0,0.0,0.0,0.0,The coffee and food are okay and there's good ...,2017-03-05 22:48:27,yelp,business


In [18]:
goodreads_samples = {}

for name, df in data.items():
    # 只保留 goodreads 的数据
    goodreads_df = df[df["source"] == "goodreads"]

    # 和上面一样，先过滤掉在 goodreads 子集里全是 NaN 的列
    goodreads_cols = goodreads_df.columns[~goodreads_df.isna().all()]

    # 只在这些有效列上取样
    sample = goodreads_df[goodreads_cols].sample(1, random_state=42)
    goodreads_samples[name] = sample

    print(f"=== {name}_goodreads sample (filtered columns) ===")
    display(sample)

=== item_goodreads sample (filtered columns) ===


,item_id,source,type,title,average_rating,description,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,kindle_asin,similar_books,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title_without_series
146246,25578275,goodreads,book,Coffee Cake Days,4.56,Meg has finally graduated and has the time she...,,25,[],US,eng,"[{'count': '56', 'name': 'to-read'}, {'count':...",,true,,[],ebook,https://www.goodreads.com/book/show/25578275-c...,"[{'author_id': '13592900', 'role': ''}]",,,29,,5,,2015,https://www.goodreads.com/book/show/25578275-c...,https://images.gr-assets.com/books/1432245860m...,29,45379896,Coffee Cake Days


=== user_goodreads sample (filtered columns) ===


,user_id,source
830493,7d6e40f200d9cc57a0bc7d61cfa7bdb9,goodreads


=== review_goodreads sample (filtered columns) ===


,review_id,user_id,item_id,stars,text,source,type,date_added,date_updated,read_at,started_at,n_votes,n_comments
4414952,57bc77dadff438129f604de55c9e0345,a835fa21459c0f5a4d59a468c0e19bd8,485651,5.0,one of my favorite books when I was younger,goodreads,book,Tue Jul 10 12:58:18 -0700 2007,Tue Jul 10 12:59:04 -0700 2007,,,0.0,0.0


In [31]:
gr_df = item_df[item_df["source"] == "goodreads"]
goodreads_cols = gr_df.columns[~gr_df.isna().all()]
gr_sample = gr_df[goodreads_cols].sample(1, random_state=42)
display(gr_sample)
print(gr_sample['popular_shelves'].values[0])

,item_id,source,type,title,average_rating,description,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,kindle_asin,similar_books,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title_without_series
146246,25578275,goodreads,book,Coffee Cake Days,4.56,Meg has finally graduated and has the time she...,,25,[],US,eng,"[{'count': '56', 'name': 'to-read'}, {'count':...",,true,,[],ebook,https://www.goodreads.com/book/show/25578275-c...,"[{'author_id': '13592900', 'role': ''}]",,,29,,5,,2015,https://www.goodreads.com/book/show/25578275-c...,https://images.gr-assets.com/books/1432245860m...,29,45379896,Coffee Cake Days


[{'count': '56', 'name': 'to-read'}, {'count': '6', 'name': 'christian-fiction'}, {'count': '4', 'name': 'contemporary'}, {'count': '3', 'name': 'young-adult'}, {'count': '2', 'name': 'kindle'}, {'count': '2', 'name': 'christian'}, {'count': '2', 'name': 'own-kindle'}, {'count': '2', 'name': 'short-stories'}, {'count': '2', 'name': 'favorites'}, {'count': '2', 'name': 'fiction'}, {'count': '1', 'name': 'currently-reading'}, {'count': '1', 'name': '5-stars'}, {'count': '1', 'name': 'read-in-2017'}, {'count': '1', 'name': 'books-for-teen-girls'}, {'count': '1', 'name': 'books-read-in-2017'}, {'count': '1', 'name': 'indie'}, {'count': '1', 'name': 'childrens'}, {'count': '1', 'name': 'children'}, {'count': '1', 'name': 'quality-christian-fiction'}, {'count': '1', 'name': 'indie-authors'}, {'count': '1', 'name': 'children-s-books'}, {'count': '1', 'name': 'my-ebook-library'}, {'count': '1', 'name': 'books-read-in-2016'}, {'count': '1', 'name': 'christian-books'}, {'count': '1', 'name': 'pc

In [ ]:
import os, json
import pandas as pd

tasks_dir = "../example/track2/goodreads/tasks"

records = []
for fname in os.listdir(tasks_dir):
    if fname.startswith("task_") and fname.endswith(".json"):
        path = os.path.join(tasks_dir, fname)
        with open(path, "r", encoding="utf-8") as f:
            t = json.load(f)
        records.append({"task_file": fname, "user_id": t["user_id"]})

df = pd.DataFrame(records)

counts = df["user_id"].value_counts()
print("unique users:", counts.shape[0])
print("users with duplicates (count > 1):", (counts > 1).sum())

dup_users = counts[counts > 1].index
df[df["user_id"].isin(dup_users)].sort_values("user_id")

unique users: 400
users with duplicates (count > 1): 0


,task_file,user_id
